# 11 — CVの採点をテスト分布に合わせる

**問題の本質:** 現行CVはベイスギ(sp15, max=298.6%)が支配し、テストには存在しない領域で採点している。  
**方針:** 学習データは変えず、評価指標だけをテスト分布(max168%, ほぼ100%以下)に合わせる。  
3指標を常に併記し、`RMSE_le170`を主判定に使う。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from src.utils import load_data, parse_spectra, get_groups, rmse, make_submission, setup_japanese_font
from src.preprocessing import snv, savitzky_golay

plt.rcParams['figure.dpi'] = 110
setup_japanese_font()
SEED = 42

# --- Data ---
train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta,  _,    X_test_raw, _ = parse_spectra(test_df)
y = y_s.values.astype(float)
groups = get_groups(train_meta)

jp_col = [c for c in train_meta.columns if c not in ['sample number','species number']][0]
sp_name_tr = train_meta.drop_duplicates('species number').set_index('species number')[jp_col].to_dict()
te_jp_col = [c for c in test_meta.columns if c not in ['sample number','species number']][0]
sp_name_te = test_meta.drop_duplicates('species number').set_index('species number')[te_jp_col].to_dict()

# --- CV splits ---
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

# --- Preprocessing ---
def preproc(X):
    return savitzky_golay(snv(X), window_length=11, polyorder=2, deriv=1)

print(f'Train: {X_raw.shape}  y: {y.min():.1f}-{y.max():.1f}%  mean={y.mean():.1f}%')
print(f'Test : {X_test_raw.shape}')
print(f'Test species: {list(sp_name_te.values())}')
print(f'Train species: {list(sp_name_tr.values())}')

## 評価指標の定義（タスク1）

| 指標 | 定義 | 用途 |
|------|------|------|
| `RMSE_all` | 全検証サンプル | 従来通り、比較の基準 |
| `RMSE_le170` | 実測≤170%のサンプルのみ | **テスト域を模した主判定指標** |
| `RMSE_clip170` | 予測を[0,170]にクリップして全サンプルで集計 | 外挿暴走・頭打ちを抑えた比較 |

In [ ]:
# ================================================================
# 評価指標
# ================================================================
def rmse_all(yt, yp):
    return float(np.sqrt(np.mean((yt - yp)**2)))

def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum() > 0 else np.nan

def rmse_clip(yt, yp, T=170.0):
    return float(np.sqrt(np.mean((yt - np.clip(yp, 0, T))**2)))

# ================================================================
# 特徴量選択ヘルパー（fold-internal, リーク厳禁）
# ================================================================
def corr_vec(X, yv):
    yc = yv - yv.mean(); Xc = X - X.mean(axis=0)
    num = (Xc * yc[:, None]).sum(0)
    den = np.sqrt((Xc**2).sum(0) * (yc**2).sum())
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num/den, 0.0)

def make_sign_selector(sign_thresh=0.8, std_thresh=0.05):
    def selector(Xtr, ytr, gtr):
        r_list = [corr_vec(Xtr[gtr==sp], ytr[gtr==sp])
                  for sp in sorted(set(gtr)) if (gtr==sp).sum()>=3]
        r_mat = np.array(r_list)
        sc = np.abs(np.sign(r_mat).sum(0)) / len(r_list)
        rs = r_mat.std(0)
        return (sc >= sign_thresh) & (rs <= std_thresh)
    return selector

# ================================================================
# CV ループ共通関数
# ================================================================
def run_cv(model_fn, feat_fn=None, use_scaler=False, label=''):
    """
    model_fn: callable() -> sklearn estimator (unfitted)
    feat_fn : callable(Xtr, ytr, gtr) -> bool array  (fold-internal selection)
    Returns : (fold_rows_list, oof_y, oof_p)
    """
    fold_rows, oof_y_list, oof_p_list = [], [], []
    for fi, (tr, va) in enumerate(SPLITS):
        Xtr = preproc(X_raw[tr]); Xva = preproc(X_raw[va])
        ytr, yva = y[tr], y[va]; gtr = groups[tr]
        n_feat = Xtr.shape[1]
        if feat_fn is not None:
            sel = feat_fn(Xtr, ytr, gtr)
            Xtr = Xtr[:, sel]; Xva = Xva[:, sel]
            n_feat = int(sel.sum())
        if use_scaler:
            sc = StandardScaler()
            Xtr = sc.fit_transform(Xtr); Xva = sc.transform(Xva)
        m = model_fn(); m.fit(Xtr, ytr)
        pred = m.predict(Xva)
        fold_rows.append({'fold': fi+1, 'n_feat': n_feat,
                          'RMSE_all':    rmse_all(yva, pred),
                          'RMSE_le170':  rmse_le(yva, pred, 170),
                          'RMSE_clip170':rmse_clip(yva, pred, 170)})
        oof_y_list.append(yva); oof_p_list.append(pred)
    oof_y = np.concatenate(oof_y_list)
    oof_p = np.concatenate(oof_p_list)
    return fold_rows, oof_y, oof_p

def print_cv_table(label, fold_rows, oof_y, oof_p):
    df = pd.DataFrame(fold_rows)
    means = df[['RMSE_all','RMSE_le170','RMSE_clip170']].mean()
    pooled_le170 = rmse_le(oof_y, oof_p, 170)
    print(f'\n--- {label} (n_feat={fold_rows[0]["n_feat"]}) ---')
    print(f'{'Fold':>6} {'RMSE_all':>10} {'RMSE_le170':>12} {'RMSE_clip170':>13}')
    for r in fold_rows:
        print(f'{r["fold"]:>6} {r["RMSE_all"]:>10.2f} {r["RMSE_le170"]:>12.2f} {r["RMSE_clip170"]:>13.2f}')
    print(f'{'Mean':>6} {means["RMSE_all"]:>10.2f} {means["RMSE_le170"]:>12.2f} {means["RMSE_clip170"]:>13.2f}')
    print(f'{'OOF':>6} {rmse_all(oof_y,oof_p):>10.2f} {pooled_le170:>12.2f}  -- (pooled)')
    return {'label': label, 'n_feat': fold_rows[0]['n_feat'],
            'RMSE_all': means['RMSE_all'],
            'RMSE_le170': means['RMSE_le170'],
            'RMSE_clip170': means['RMSE_clip170'],
            'RMSE_le170_pooled': pooled_le170}

print('Metric functions and run_cv defined.')
print(f'Samples with y<=170%: {(y<=170).sum()}/{len(y)} ({100*(y<=170).mean():.1f}%)')
print(f'Samples with y>170%:  {(y>170).sum()} (テスト域外、採点から外す)')

## タスク2: 現行ET(全波数)を新指標で測り直す

これがテストに近い正直な基準値になる。`RMSE_le170`を以降の改善判定の主役にする。

In [ ]:
ET_KW = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

print('Running ET (all wavenumbers, SNV+SG1)...')
et_rows, et_oof_y, et_oof_p = run_cv(
    model_fn=lambda: ExtraTreesRegressor(**ET_KW),
    label='ET 全波数'
)
et_summary = print_cv_table('ET 全波数 (baseline)', et_rows, et_oof_y, et_oof_p)

print(f'\n>>> テストに近い基準値 (RMSE_le170 fold平均) = {et_summary["RMSE_le170"]:.2f}%')
print(f'>>> 従来基準 (RMSE_all fold平均)              = {et_summary["RMSE_all"]:.2f}%')
print(f'>>> 改善して見えていた差の多くはFold3の高含水によるもの')

# サンプル数内訳
n_le170 = (et_oof_y <= 170).sum()
n_total = len(et_oof_y)
print(f'\nRMSE_le170 の対象サンプル数: {n_le170}/{n_total} ({100*n_le170/n_total:.1f}%)')

# Fold別の「RMSE_all vs RMSE_le170」可視化
df_et = pd.DataFrame(et_rows)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(5)
w = 0.3
axes[0].bar(x-w/2, df_et['RMSE_all'],     w, label='RMSE_all',     color='steelblue',  alpha=0.7)
axes[0].bar(x+w/2, df_et['RMSE_le170'],   w, label='RMSE_le170',   color='darkorange',  alpha=0.7)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'Fold{i+1}' for i in range(5)])
axes[0].set_ylabel('RMSE (%)')
axes[0].set_title('Fold別: RMSE_all vs RMSE_le170\n(高含水サンプルを採点から外すと?)')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# 散布図: 予測 vs 実測 (RMSE_le170 の対象域を強調)
m_le = et_oof_y <= 170; m_gt = ~m_le
axes[1].scatter(et_oof_y[m_le], et_oof_p[m_le], s=5, alpha=0.3,
               color='darkorange', label=f'y<=170% (n={m_le.sum()}, RMSE_le170に含む)')
axes[1].scatter(et_oof_y[m_gt], et_oof_p[m_gt], s=5, alpha=0.3,
               color='gray',      label=f'y>170%  (n={m_gt.sum()}, テスト域外)')
lim = max(et_oof_y.max(), et_oof_p.max()) * 1.02
axes[1].plot([0,lim],[0,lim],'r--',lw=1)
axes[1].axhline(170, color='navy', lw=1, ls=':', label='y=170% 閾値')
axes[1].axvline(170, color='navy', lw=1, ls=':')
axes[1].set_xlabel('実測含水率 (%)'); axes[1].set_ylabel('予測含水率 (%)')
axes[1].set_title('OOF予測 vs 実測\n(橙=RMSE_le170対象域)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/t2_baseline_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## タスク3: 予測値クリップの効果

後処理として `pred = min(pred, T)` を適用し、各閾値でRMSEへの影響を測定する。  
ET は元々191%で頭打ちなので効果は限定的かもしれないが、提出時の安全弁として評価する。

In [ ]:
# ET のOOFはタスク2で計算済み → et_oof_y, et_oof_p を再利用
T_vals = [130, 150, 170, 200, 216, 9999]  # 9999 = no clip

print('=== クリップ閾値 T別 RMSE (ET全波数 OOF) ===')
print(f'{"T":>6} {"pred_max":>9} {"RMSE_all":>10} {"RMSE_le170":>12} {"RMSE_clip":>11}')
print('-' * 55)

clip_rows = []
for T in T_vals:
    p_clipped = np.clip(et_oof_p, 0, T)
    ra  = rmse_all(et_oof_y, p_clipped)
    rle = rmse_le(et_oof_y,  p_clipped, 170)   # le170 は予測クリップの影響を受ける
    rc  = rmse_clip(et_oof_y, et_oof_p, T)     # 原定義: clipして集計
    T_label = f'{T}%' if T < 9000 else 'None'
    print(f'{T_label:>6} {p_clipped.max():>9.1f} {ra:>10.2f} {rle:>12.2f} {rc:>11.2f}')
    clip_rows.append({'T': T_label, 'RMSE_all': ra, 'RMSE_le170': rle, 'RMSE_clip': rc})

print('\nNote: RMSE_le170 は y<=170 のサンプルのみ。クリップが逆効果になることも確認。')

# プロット
fig, ax = plt.subplots(figsize=(9, 5))
T_num = [130, 150, 170, 200, 216, 999]
T_lbl = ['130', '150', '170', '200', '216(訓練上限)', 'none']
ra_vals  = [np.clip(et_oof_p,0,T) for T in T_num]
rmse_a_vals   = [rmse_all(et_oof_y, np.clip(et_oof_p,0,T)) for T in T_num]
rmse_le_vals  = [rmse_le(et_oof_y,  np.clip(et_oof_p,0,T)) for T in T_num]
ax.plot(T_lbl, rmse_a_vals,  'o-', label='RMSE_all (クリップ後)', color='steelblue')
ax.plot(T_lbl, rmse_le_vals, 's-', label='RMSE_le170 (クリップ後)', color='darkorange')
ax.set_xlabel('クリップ上限 T (%)')
ax.set_ylabel('RMSE (%)')
ax.set_title('クリップ閾値 vs RMSE (ET全波数 OOF)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/t3_clipping_effect.png', dpi=150, bbox_inches='tight')
plt.show()

# 最適クリップ閾値の特定
best_T_idx = int(np.argmin(rmse_a_vals))
print(f'\n最も RMSE_all を下げるクリップ閾値: T={T_lbl[best_T_idx]}% => {rmse_a_vals[best_T_idx]:.2f}%')
print(f'クリップなし(T=none) RMSE_all: {rmse_a_vals[-1]:.2f}%')

## タスク4: 新指標のもとでの改善再評価

**主判定: `RMSE_le170`**（テスト域相当）を使い、1要素ずつ変える。  
特徴選択は全てfold内学習側のみで計算（リーク厳禁）。

In [ ]:
# 比較対象
# (1) ET  + 全波数         [Task2で実施済み]
# (2) ET  + 一貫性選択(sign0.8, std0.05)  fold-internal
# (3) RF  + 全波数
# (4) HistGB + 全波数
# (5) Ridge + 全波数 + scaler  (RMSE_clip が意味を持つ代表例)

RF_KW  = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

configs = [
    ('ET 全波数 [baseline]',
     lambda: ExtraTreesRegressor(**ET_KW),
     None, False),
    ('ET 一貫性選択(sign>=0.8)',
     lambda: ExtraTreesRegressor(**ET_KW),
     make_sign_selector(0.8, 0.05), False),
    ('RF 全波数',
     lambda: RandomForestRegressor(**RF_KW),
     None, False),
    ('HistGB 全波数',
     lambda: HistGradientBoostingRegressor(max_iter=300, random_state=SEED),
     None, False),
    ('Ridge 全波数 (外挿検証)',
     lambda: Ridge(alpha=100),
     None, True),
]

summary_rows = []
oof_store = {}  # label -> (oof_y, oof_p)

# ET全波数 は既計算
summary_rows.append(et_summary)
oof_store['ET 全波数 [baseline]'] = (et_oof_y, et_oof_p)

for label, mfn, ffn, scl in configs[1:]:
    print(f'Running {label}...')
    rows, oy, op = run_cv(mfn, feat_fn=ffn, use_scaler=scl)
    s = print_cv_table(label, rows, oy, op)
    summary_rows.append(s)
    oof_store[label] = (oy, op)

print('\nAll configurations done.')

In [ ]:
# ---- 比較サマリー表 ----
print('\n' + '='*80)
print('モデル比較サマリー (RMSE_le170 = テスト域指標, 主判定)')
print('='*80)
df_sum = pd.DataFrame(summary_rows)
df_sum_disp = df_sum[['label','n_feat','RMSE_all','RMSE_le170','RMSE_clip170']].copy()
df_sum_disp = df_sum_disp.sort_values('RMSE_le170')
print(df_sum_disp.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

baseline_le170 = et_summary['RMSE_le170']
print(f'\n基準値 (ET全波数 RMSE_le170): {baseline_le170:.2f}%')
for r in summary_rows[1:]:
    delta = r['RMSE_le170'] - baseline_le170
    tag = 'BETTER' if delta < -0.3 else ('worse' if delta > 0.3 else 'tie')
    print(f'  {r["label"]:35s}: {r["RMSE_le170"]:.2f}% ({delta:+.2f}%) [{tag}]')

# グラフ
fig, ax = plt.subplots(figsize=(11, 5))
labels = [r['label'] for r in summary_rows]
x = np.arange(len(labels))
w = 0.28
ax.bar(x-w, [r['RMSE_all']    for r in summary_rows], w, label='RMSE_all',     color='steelblue',  alpha=0.7)
ax.bar(x,   [r['RMSE_le170']  for r in summary_rows], w, label='RMSE_le170',   color='darkorange',  alpha=0.9)
ax.bar(x+w, [r['RMSE_clip170']for r in summary_rows], w, label='RMSE_clip170', color='mediumseagreen', alpha=0.7)
ax.axhline(baseline_le170, color='darkorange', lw=2, ls='--', label=f'ET_all RMSE_le170={baseline_le170:.1f}%')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=18, ha='right', fontsize=9)
ax.set_ylabel('RMSE (%)')
ax.set_title('モデル×特徴の3指標比較\n(オレンジ棒=主判定 RMSE_le170)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/t4_model_comparison_new_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## タスク5（診断）: テスト樹種のスペクトル位置付け

テスト各樹種の予測分布と、訓練データとの含水率域の対応を確認。  
「テストに高含水がない」ことの裏付けと、樹種ごとの予測妥当性チェック。

In [ ]:
# --- 全訓練データで ET を学習してテスト予測 ---
X_tr_pp  = preproc(X_raw)
X_te_pp  = preproc(X_test_raw)

et_full = ExtraTreesRegressor(**ET_KW)
et_full.fit(X_tr_pp, y)
te_pred = et_full.predict(X_te_pp)

te_groups = test_meta['species number'].values
te_sp_list = sorted(set(te_groups))

print('=== テスト予測値 (ET, 全訓練データで学習) ===')
print(f'{"樹種":>20} {"n":>5} {"min":>7} {"max":>7} {"mean":>7} {"p95":>7} {"<=100%":>8}')
print('-' * 62)
for sp in te_sp_list:
    m = te_groups == sp
    p = te_pred[m]
    print(f'{sp_name_te.get(sp,str(sp)):>20} {m.sum():>5} '
          f'{p.min():>7.1f} {p.max():>7.1f} {p.mean():>7.1f} '
          f'{np.percentile(p,95):>7.1f} {(p<=100).mean()*100:>7.1f}%')

print(f'\nテスト全体: min={te_pred.min():.1f}  max={te_pred.max():.1f}  '
      f'mean={te_pred.mean():.1f}  p95={np.percentile(te_pred,95):.1f}%')
print(f'テスト予測で >170%: {(te_pred>170).sum()} / {len(te_pred)}')
print(f'テスト予測で >100%: {(te_pred>100).sum()} / {len(te_pred)}')

# ヒストグラム: テスト予測の分布 (樹種別)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for sp in te_sp_list:
    m = te_groups == sp
    axes[0].hist(te_pred[m], bins=15, alpha=0.6,
                label=f'sp{sp} {sp_name_te.get(sp,"")} (n={m.sum()})')
axes[0].axvline(170, color='red', lw=1.5, ls='--', label='170% 閾値')
axes[0].set_xlabel('予測含水率 (%)')
axes[0].set_ylabel('件数')
axes[0].set_title('テストデータ: 樹種別予測分布')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 訓練 vs テストの予測分布比較
axes[1].hist(et_oof_p, bins=40, alpha=0.4, color='steelblue',
            label=f'訓練(OOF) mean={et_oof_p.mean():.0f}%', density=True)
axes[1].hist(te_pred,  bins=30, alpha=0.6, color='darkorange',
            label=f'テスト    mean={te_pred.mean():.0f}%', density=True)
axes[1].axvline(170, color='red', lw=1.5, ls='--', label='170% 閾値')
axes[1].set_xlabel('予測含水率 (%)')
axes[1].set_ylabel('密度')
axes[1].set_title('予測分布: 訓練OOF vs テスト\n(テストが訓練より低含水域に集中?)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/t5_test_spectrum_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 訓練樹種とのスペクトル距離 ---
print('\n=== テスト樹種のスペクトル vs 訓練含水率帯域の平均距離 (L2) ===')
tr_bands = [(0,50,'0-50%'), (50,100,'50-100%'), (100,200,'100-200%'), (200,400,'200-400%')]
tr_band_mean = {}
for lo, hi, lbl in tr_bands:
    m = (y >= lo) & (y < hi)
    if m.sum() > 0:
        tr_band_mean[lbl] = X_tr_pp[m].mean(0)

print(f'{"テスト樹種":>15}', end='')
for lbl in tr_band_mean: print(f'{lbl:>14}', end='')
print(f'{"最近傍帯域":>15}')
print('-' * (15 + 14*len(tr_band_mean) + 15))

for sp in te_sp_list:
    m = te_groups == sp
    sp_mean = X_te_pp[m].mean(0)
    dists = {lbl: float(np.linalg.norm(sp_mean - mu))
             for lbl, mu in tr_band_mean.items()}
    nearest = min(dists, key=dists.get)
    print(f'{sp_name_te.get(sp,str(sp)):>15}', end='')
    for v in dists.values(): print(f'{v:>14.4f}', end='')
    print(f'{nearest:>15}')

## 総括

In [ ]:
print('=' * 72)
print('総括')
print('=' * 72)

print('\n[1] テストに近い正直な基準値]')
best_r = min(summary_rows, key=lambda r: r['RMSE_le170'])
print(f'  ET全波数  RMSE_le170 = {et_summary["RMSE_le170"]:.2f}%  (従来RMSE_all={et_summary["RMSE_all"]:.2f}%)')

print('\n[2] クリップの安全弁効果]')
best_clip_T_idx = int(np.argmin([rmse_clip(et_oof_y, et_oof_p, T) for T in [130,150,170,200,216]]))
best_T_list = [130,150,170,200,216]
print(f'  RMSE_all を最小化するクリップ T = {best_T_list[best_clip_T_idx]}%')
print(f'  テスト提出への推奨: 予測を [0, {best_T_list[best_clip_T_idx]}] にクリップ')

print('\n[3] RMSE_le170 で現行を上回った設定]')
winners = [r for r in summary_rows
           if r['RMSE_le170'] < et_summary['RMSE_le170'] - 0.3]
if winners:
    for r in sorted(winners, key=lambda r: r['RMSE_le170']):
        delta = r['RMSE_le170'] - et_summary['RMSE_le170']
        print(f'  {r["label"]:38s}: {r["RMSE_le170"]:.2f}% ({delta:+.2f}%) *** 提出候補')
else:
    print('  今回の範囲では有意な改善なし')

print('\n[4] 次に詰めるべき方向]')
print('  RMSE_le170 の改善幅が最大だった軸を以下で確認:')
df_s = pd.DataFrame(summary_rows)[['label','RMSE_le170']].sort_values('RMSE_le170')
print(df_s.to_string(index=False, float_format=lambda x: f'{x:.2f}'))